In [ ]:
import cv2
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Qwen2TokenizerFast, Qwen3ForCausalLM, AutoConfig

from diffusers.utils import load_image
from diffusers.schedulers import FlowMatchEulerDiscreteScheduler
from diffusers.models import AutoencoderKLFlux2, Flux2Transformer2DModel

from ctgmworkshop.flux_tools import *
from ctgmworkshop.image_tools import *

plt.style.use("dark_background")

In [5]:
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
    "../../FLUX.2-klein-4B/scheduler", device=device
)

In [2]:
all_cond_latents_A = torch.load(
    "../../precomputes/flow_refractor/test-7/all_cond_latents_A.pt"
)
all_cond_latents_B = torch.load(
    "../../precomputes/flow_refractor/test-7/all_cond_latents_B.pt"
)
all_final_latents_B = torch.load(
    "../../precomputes/flow_refractor/test-7/all_final_latents_B.pt"
)

In [4]:
from ctgmworkshop.architectures.flow_refractor.scaled_up_deeperflow import (
    FlowRefractorModel,
)

flow_refractor = FlowRefractorModel()
flow_refractor = flow_refractor.to(device)
flow_refractor.eval()

total_params = sum(p.numel() for p in flow_refractor.parameters())
print("Total params:", total_params / 1000000, "M")

state_dict = torch.load(
    "../../trained_models/flow_refractor/run_19_flow_refractor_l.pth",
    weights_only=True,
)
flow_refractor.load_state_dict(state_dict)

Total params: 152.571088 M


<All keys matched successfully>

In [6]:
@torch.no_grad()
def refract(init, cond_A, cond_B, final_A):
    scheduler.set_timesteps(16, mu=1.0)
    latents = init
    flow_refractor.eval()

    for t in scheduler.timesteps:
        latent_model_input = latents
        t = t.to(device).view(1)
        predicted_noise = flow_refractor(
            sample=latent_model_input,
            timestep=t,
            frame_A=cond_A,
            frame_B=cond_B,
            edited_frame_A=final_A,
        )

        latents = scheduler.step(predicted_noise, t, latents).prev_sample

    return latents

In [7]:
all_final_latents_A = torch.zeros_like(all_final_latents_B)
all_init = torch.zeros_like(all_final_latents_B)

In [8]:
num_samples = all_cond_latents_A.shape[0]
batch_size = 20

for i in tqdm(range(num_samples // batch_size)):
    cond_A = all_cond_latents_A[i * batch_size : (i + 1) * batch_size, ...]
    cond_B = all_cond_latents_B[i * batch_size : (i + 1) * batch_size, ...]
    final_B = all_final_latents_B[i * batch_size : (i + 1) * batch_size, ...]

    cond_A = cond_A.cuda().float()
    cond_B = cond_B.cuda().float()
    final_B = final_B.cuda().float()

    init = torch.normal(0, 1, cond_A.shape).to(cond_A.device, cond_A.dtype)

    final_A = refract(init, cond_A=cond_B, cond_B=cond_A, final_A=final_B)  # swapped!

    all_final_latents_A[i * batch_size : (i + 1) * batch_size, ...] = final_A
    all_init[i * batch_size : (i + 1) * batch_size, ...] = init


100%|██████████| 1000/1000 [51:21<00:00,  3.08s/it]


In [ ]:
torch.save(
    all_final_latents_A,
    "../../precomputes/flow_refractor/test-7/all_final_latents_A_rect.pt",
)
torch.save(all_init, "../../precomputes/flow_refractor/test-7/all_init_rect.pt")